# Backtest Visualization

This notebook visualizes backtest results including equity curves, trade markers, drawdowns, and performance metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

# Project imports
import sys
sys.path.insert(0, '..')
from env.trading_env import TradingEnv
from env.preprocessor import load_and_preprocess, load_npy, preprocess

## Load Backtest Results

In [ ]:
# Find latest evaluation results
eval_dir = Path('../data/evaluation')

if eval_dir.exists():
    summary_files = sorted(eval_dir.glob('summary_*.json'))
    if summary_files:
        latest_summary = summary_files[-1]
        with open(latest_summary) as f:
            summary = json.load(f)
        print(f'Loaded summary from: {latest_summary.name}')
    else:
        print('No evaluation results found. Run evaluation first.')
        summary = None
else:
    print('Evaluation directory not found.')
    summary = None

# Load trades if available
trades_files = sorted(eval_dir.glob('trades_*.csv')) if eval_dir.exists() else []
if trades_files:
    trades_df = pd.read_csv(trades_files[-1])
    print(f'Loaded {len(trades_df)} trades')
else:
    trades_df = None

# Load equity curve
equity_files = sorted(eval_dir.glob('equity_curve_*.npy')) if eval_dir.exists() else []
if equity_files:
    equity_curve = np.load(equity_files[-1])
    print(f'Equity curve length: {len(equity_curve)}')
else:
    equity_curve = None

## Display Metrics

In [ ]:
if summary and 'metrics' in summary:
    metrics = summary['metrics']
    
    print('=' * 50)
    print('BACKTEST PERFORMANCE METRICS')
    print('=' * 50)
    print(f"\nTotal Trades: {metrics.get('total_trades', 'N/A')}")
    print(f"Winning Trades: {metrics.get('winning_trades', 'N/A')}")
    print(f"Losing Trades: {metrics.get('losing_trades', 'N/A')}")
    print(f"Win Rate: {metrics.get('win_rate', 0) * 100:.2f}%")
    print(f"\nTotal P&L: ${metrics.get('total_pnl', 0):.2f}")
    print(f"Average P&L: ${metrics.get('avg_pnl', 0):.2f}")
    print(f"Average Win: ${metrics.get('avg_win', 0):.2f}")
    print(f"Average Loss: ${metrics.get('avg_loss', 0):.2f}")
    print(f"Profit Factor: {metrics.get('profit_factor', 0):.2f}")
    print(f"\nFinal Balance: ${metrics.get('final_balance', 0):.2f}")
    print(f"Total Return: {metrics.get('total_return_pct', 0):.2f}%")
    print(f"Sharpe Ratio: {metrics.get('sharpe_ratio', 0):.2f}")
    print(f"Max Drawdown: ${metrics.get('max_drawdown', 0):.2f} ({metrics.get('max_drawdown_pct', 0):.2f}%)")
    print(f"Calmar Ratio: {metrics.get('calmar_ratio', 0):.2f}")
    print(f"Expectancy: {metrics.get('expectancy', 0):.2f}")
    print('=' * 50)

## Equity Curve

In [ ]:
if equity_curve is not None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # Equity curve
    axes[0].plot(equity_curve, linewidth=1)
    axes[0].axhline(y=equity_curve[0], color='gray', linestyle='--', alpha=0.5, label='Initial Balance')
    axes[0].set_title('Equity Curve')
    axes[0].set_ylabel('Equity ($)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Drawdown
    cumulative = np.cumsum([0] + list(np.diff(equity_curve))) + equity_curve[0]
    running_max = np.maximum.accumulate(cumulative)
    drawdown = (running_max - cumulative) / running_max * 100
    
    axes[1].fill_between(range(len(drawdown)), 0, -drawdown, alpha=0.5, color='red')
    axes[1].plot(-drawdown, linewidth=1, color='red')
    axes[1].set_title('Drawdown')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].set_xlabel('Step')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Trade Analysis

In [ ]:
if trades_df is not None and len(trades_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # P&L distribution
    axes[0, 0].hist(trades_df['pnl'], bins=30, edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[0, 0].set_title('P&L Distribution')
    axes[0, 0].set_xlabel('P&L ($)')
    axes[0, 0].set_ylabel('Frequency')
    
    # Cumulative P&L
    cumulative_pnl = trades_df['pnl'].cumsum()
    axes[0, 1].plot(cumulative_pnl, linewidth=1)
    axes[0, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[0, 1].set_title('Cumulative P&L')
    axes[0, 1].set_xlabel('Trade Number')
    axes[0, 1].set_ylabel('Cumulative P&L ($)')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Trade exit reasons
    if 'exit_reason' in trades_df.columns:
        exit_counts = trades_df['exit_reason'].value_counts()
        axes[1, 0].pie(exit_counts.values, labels=exit_counts.index, autopct='%1.1f%%')
        axes[1, 0].set_title('Exit Reasons')
    
    # P&L by trade index
    axes[1, 1].scatter(range(len(trades_df)), trades_df['pnl'], 
                       c=['green' if p > 0 else 'red' for p in trades_df['pnl']],
                       alpha=0.6, s=50)
    axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, 1].set_title('Individual Trade P&L')
    axes[1, 1].set_xlabel('Trade Number')
    axes[1, 1].set_ylabel('P&L ($)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Rolling Metrics

In [ ]:
if trades_df is not None and len(trades_df) > 0:
    # Calculate rolling win rate (20-trade window)
    window = 20
    rolling_win = (trades_df['pnl'] > 0).rolling(window).mean() * 100
    
    # Calculate rolling P&L
    rolling_pnl = trades_df['pnl'].rolling(window).sum()
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    axes[0].plot(rolling_win, linewidth=1)
    axes[0].axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_title(f'Rolling Win Rate ({window}-trade window)')
    axes[0].set_ylabel('Win Rate (%)')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(rolling_pnl, linewidth=1, color='green')
    axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_title(f'Rolling P&L ({window}-trade window)')
    axes[1].set_ylabel('P&L ($)')
    axes[1].set_xlabel('Trade Number')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Run Live Backtest

In [ ]:
from stable_baselines3 import PPO

# Load model
model_dir = Path('../models')
model_files = sorted(model_dir.glob('ppo_*.zip'))

if model_files:
    model_path = model_files[-1]
    print(f'Loading model from: {model_path}')
    model = PPO.load(model_path)
    
    # Load and preprocess data
    data_path = Path('../data/raw/EURUSD.csv')
    
    if data_path.exists():
        processed = load_and_preprocess(data_path)
        print(f'Data shape: {processed.shape}')
        
        # Create environment
        env = TradingEnv(
            data=processed,
            window_size=10,
            initial_balance=10000.0,
            spread=0.0001,
            slippage_prob=0.0,
        )
        
        # Run backtest
        obs, _ = env.reset()
        
        equity_curve_demo = [env.equity]
        trades_demo = []
        
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            equity_curve_demo.append(env.equity)
            
            if info.get('closed_trade'):
                trade = info['closed_trade']
                trades_demo.append({
                    'step': len(trades_demo),
                    'pnl': trade.pnl,
                    'exit_reason': trade.exit_reason,
                })
            
            done = terminated or truncated
        
        print(f'Completed {len(equity_curve_demo)} steps')
        print(f'Trades executed: {len(trades_demo)}')
        print(f'Final equity: ${env.equity:.2f}')
        
        # Plot results
        fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        
        axes[0].plot(equity_curve_demo, linewidth=1)
        axes[0].axhline(y=10000, color='gray', linestyle='--', alpha=0.5, label='Initial')
        axes[0].set_title('Backtest - Equity Curve')
        axes[0].set_ylabel('Equity ($)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Mark trades
        if trades_demo:
            trade_pnls = [t['pnl'] for t in trades_demo]
            axes[1].scatter(range(len(trades_demo)), trade_pnls, 
                           c=['green' if p > 0 else 'red' for p in trade_pnls],
                           s=100, marker='v', alpha=0.7)
        axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        axes[1].set_title('Trade Outcomes')
        axes[1].set_ylabel('P&L ($)')
        axes[1].set_xlabel('Trade Number')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print('No trained model found. Run training first.')